# Verify Hardware-Realized QuAK Algorithm

This notebook creates hardware-realizable variants of `QuantumNativeQNetwork` and `QuantumNativeQNetworkConv` model instances by overriding the quantum circuit, and loads any pre-trained model parameters from file. The notebook then compares the output of original pre-trained model and its hardware-realized variant over many random environment observation samples to ensure the models produce the same exact action-probability distribution for each environment observation.

In [1]:
from dataclasses import dataclass
from functools import partial

from flax import nnx
import jax
import jax.numpy as jnp
import pennylane as qp

from qnrl.modules.quantumnativeqnetwork import circuit_layer_reset, prepare_returns_probability_distribution, QuantumNativeQNetwork
from qnrl.modules.quantumnativeqnetworkconv import QuantumNativeQNetworkConv


def circuit_action_dist_hardware_realizable(
    inputs: jax.Array, # (n_actions, n_layers, d_returns, num_enc_features,)
    probs_actions: jax.Array,
    weights: jax.Array, # (n_actions, n_layers, d_returns, num_var_features,)
    n_actions: int,
    d_actions: int,
    d_returns: int,
    m_moment: int,
    n_layers: int,
    U_atoms: jax.Array,
    pauli_enc: tuple[str, ...],
    pauli_var: tuple[str, ...],
    imprimitive: str | None, # Entanglement gate string.
    use_qft: bool,
    ):
    """Quantum circuit for estimating the action distribution via m-th moment of return distribuion followed by amplitude kickback trick with respect to an input observation.
    
    This circuit uses block encoding with ancilla qubits and post-selection to encode a trace non-increasing matrix `U_atoms`.
    """
    
    # Create wires.
    wiredict = {}
    wiredict['actions'] = list(range(d_actions))
    wiredict['ancilla_m'] = list(range(wiredict['actions'][-1]+1, wiredict['actions'][-1]+1+m_moment))
    wiredict['ancilla_block'] = wiredict['ancilla_m'][-1] + 1
    wiredict['returns'] = list(range(wiredict['ancilla_block']+1, wiredict['ancilla_block']+1+d_returns))
    
    # Weighted superposition of all actions.
    # Use `StatePrep` here because there may be fewer "active" actions than the total number of actions supported by the number of qubits in the action register. This ensures that all "inactive" actions have 0 amplitude.
    qp.StatePrep(probs_actions, wires=wiredict['actions'], normalize=True, pad_with=0)
    
    # Prepare returns probability distribution for every policy.
    prepare_returns_probability_distribution(
        inputs=inputs,
        weights=weights,
        n_actions=n_actions,
        n_layers=n_layers,
        wires_actions=wiredict['actions'],
        wires_returns=wiredict['returns'],
        pauli_enc=pauli_enc,
        pauli_var=pauli_var,
        imprimitive=imprimitive,
        use_qft=use_qft,
    )
    
    # Prepare the m-th moment of the returns distribution via block encoding of the return-atom Kraus matrix.
    for idx in range(m_moment):
        qp.BlockEncode(U_atoms, wires=[
            wiredict['ancilla_block'], # Ancilla first,
            *wiredict['returns'], # Then all the application wires.
        ])
        qp.CNOT(wires=[
            wiredict['ancilla_block'], # Control wire.
            wiredict['ancilla_m'][idx], # Target wire.
        ])
    
    # Optionally apply a mid-circuit measurement to collapse the ancilla wires to either |0> or |1>.
    # Depending on the ancilla result, the state on the system wires will be:
    # - If ancilla = |0>, then |system> = U_vz |psi>
    # - If ancilla = |1>, then |system> = garbage
    for idx in range(m_moment):
        qp.measure(wiredict['ancilla_m'][idx], postselect=0, reset=True)
    
    # Measure the return-atom qubits to collapse their state, then reset to zero.
    circuit_layer_reset(wires=wiredict['returns'])
    
    # Generate action distribuion via measurements across the action wires.
    return qp.probs(wires=wiredict['actions'])


@partial(nnx.jit, static_argnames=[
    # Device and QNode parameters.
    'device_name', 'shots', 'diff_method',
    # Circuit parameters.
    'n_actions',
    'd_actions',
    'd_returns',
    'm_moment',
    'n_layers',
    'U_atoms',
    'pauli_enc',
    'pauli_var',
    'imprimitive',
    'use_qft',
])
def run_circuit_action_dist_hardware_realizable(
    # All device and QNode parameters.
    key: jax.Array,
    device_name: str,
    shots: int | None,
    diff_method: str | None,
    # Circuit arguments.
    inputs: jax.Array, # (n_actions, n_layers, d_returns, num_enc_features,)
    weights: jax.Array, # (n_actions, n_layers, d_returns, num_var_features,)
    probs_actions: jax.Array,
    n_actions: int,
    d_actions: int,
    d_returns: int,
    m_moment: int,
    n_layers: int,
    U_atoms: jax.Array,
    pauli_enc: tuple[str, ...],
    pauli_var: tuple[str, ...],
    imprimitive: str | None, # Entanglement gate string.
    use_qft: bool,
    ):
    """Run `circuit_action_dist` on a single input instance."""
    device = qp.device(name=device_name, shots=shots, seed=key)
    qnode = qp.QNode(func=circuit_action_dist_hardware_realizable, device=device, interface="jax", diff_method=diff_method)
    return qnode(
        inputs=inputs,
        probs_actions=probs_actions,
        weights=weights,
        n_actions=n_actions,
        d_actions=d_actions,
        d_returns=d_returns,
        m_moment=m_moment,
        n_layers=n_layers,
        U_atoms=U_atoms,
        pauli_enc=pauli_enc,
        pauli_var=pauli_var,
        imprimitive=imprimitive,
        use_qft=use_qft,
    )


@partial(nnx.jit, static_argnames=[
    # Device and QNode parameters.
    'device_name', 'shots', 'diff_method',
    # Circuit parameters.
    'n_actions',
    'd_actions',
    'd_returns',
    'm_moment',
    'n_layers',
    'U_atoms',
    'pauli_enc',
    'pauli_var',
    'imprimitive',
    'use_qft',
])
def batched_run_circuit_action_dist_hardware_realizable(
    # All device and QNode parameters.
    key: jax.Array,
    device_name: str,
    shots: int | None,
    diff_method: str | None,
    # Circuit arguments.
    inputs: jax.Array, # (n_actions, n_layers, d_returns, num_enc_features,)
    weights: jax.Array, # (n_actions, n_layers, d_returns, num_var_features,)
    probs_actions: jax.Array,
    n_actions: int,
    d_actions: int,
    d_returns: int,
    m_moment: int,
    n_layers: int,
    U_atoms: jax.Array,
    pauli_enc: tuple[str, ...],
    pauli_var: tuple[str, ...],
    imprimitive: str | None, # Entanglement gate string.
    use_qft: bool,
    ):
    """Run `circuit_action_dist` on a batch of input instances."""
    # Create vmap of core function.
    mapped_func = nnx.vmap(
        f=run_circuit_action_dist_hardware_realizable, 
        in_axes=(
            0, # key
            None, # device_name
            None, # shots
            None, # diff_method
            0, # inputs
            None, # weights
            0, # probs_actions
            None, # n_actions
            None, # d_actions
            None, # d_returns
            None, # m_moment
            None, # n_layers
            None, # U_atoms
            None, # pauli_enc
            None, # pauli_var
            None, # imprimitive
            None, # use_qft
            ),
        )
    # Call mapped function with manually assigned positional arguments.
    return mapped_func(
        key,
        device_name,
        shots,
        diff_method,
        inputs,
        weights,
        probs_actions,
        n_actions,
        d_actions,
        d_returns,
        m_moment,
        n_layers,
        U_atoms,
        pauli_enc,
        pauli_var,
        imprimitive,
        use_qft,
    )


@dataclass(kw_only=True)
class QuantumNativeQNetworkHardwareRealized(QuantumNativeQNetwork):
    """Clone of `QuantumNativeQNetwork` that uses hardware-realizable variant of QuAK circuit."""
    
    @classmethod
    def monkeypatch_from_QuantumNativeQNetwork(cls, parent: QuantumNativeQNetwork) -> 'QuantumNativeQNetworkHardwareRealized':
        """Instantiate an instance of the `QuantumNativeQNetworkHardwareRealized` class using members from a base `QuantumNativeQNetwork` class.
        
        This is useful to reuse existing trained model parameters.
        """
        # Create new child instance for monkeypatching.
        monkey = cls(
            din=parent.din,
            n_actions=parent.n_actions,
            n_returns=parent.n_returns,
            z_min=parent.z_min,
            z_max=parent.z_max,
            m_moment=parent.m_moment,
            n_reuploads=parent.n_reuploads,
            n_layers=parent.n_layers,
            rngs_kwargs=parent.rngs_kwargs,
            pauli_enc=parent.pauli_enc,
            pauli_var=parent.pauli_var,
            imprimitive=parent.imprimitive,
            use_qft=parent.use_qft,
            input_preprocess=parent.input_preprocess,
            input_multiplex_activation=parent.input_multiplex_activation,
        )
        
        # Extract the graph state and variables from the parent and update the monkey child.
        graphdef, state = nnx.split(parent)
        nnx.update(monkey, state)
        
        # Return the updated monkey child.
        return monkey
    
    @nnx.jit(static_argnames=['n_reuploads'])
    def call_gen_action_dist(self, x, n_reuploads: int | None = None): # (batch,din,)
        """Generate action distribuion from input observation using hardware-realized QuAK circuit."""
        
        # Project input observation into shape that is compatible with the quantum circuit.
        x = self.project_observations(x) # (batch, n_layers, d_returns, 1,)
        
        # Track the batch size.
        batch_size = x.shape[0]
        
        # Prepare the n-th power of the m-th moment of the returns distribution.
        key = self.rngs.default() # Generate new random key for circuit measurement.
        # Initially set all actions to equal probability (here all have magnitude of 1, which will be normalized within the quantum circuit code later).
        probs_actions = jnp.ones(shape=(batch_size, self.n_actions,), dtype='float64')
        if n_reuploads is None:
            n_reuploads = self.n_reuploads
        # Create a batch of keys.
        subkeys = [key]
        for _ in range(n_reuploads):
            # Create a batch of keys for the current reupload.
            subkeys = jax.random.split(key=subkeys[0], num=batch_size)
            # Run the circuit on batched input.
            probs_actions = batched_run_circuit_action_dist_hardware_realizable( # (batch,2**d_actions,)
                # All device and QNode parameters.
                key=subkeys,
                device_name="default.qubit",
                shots=None,
                diff_method=None,
                # Circuit arguments.
                inputs=x,
                probs_actions=probs_actions,
                weights=self.circ_weights_var,
                n_actions=self.n_actions,
                d_actions=self.d_actions,
                d_returns=self.d_returns,
                m_moment=self.m_moment,
                n_layers=self.n_layers,
                U_atoms=self.U_atoms,
                pauli_enc=self.pauli_enc,
                pauli_var=self.pauli_var,
                imprimitive=self.imprimitive,
                use_qft=self.use_qft,
            )
            # Filter circuit output based on number of "active" actions.
            # If desired number of actions is not a multiple of 2, then filter out the excess actions on the qubit register (i.e., more actions can be represented on the register and we don't care about them).
            probs_actions = probs_actions[:,jnp.array(self.active_action_idx, dtype=int)] # (batch,n_actions,)
            # Ensure probabilities are normalized after the filter.
            probs_actions = probs_actions / probs_actions.sum(axis=-1, keepdims=True)
        return probs_actions


@dataclass(kw_only=True)
class QuantumNativeQNetworkConvHardwareRealized(QuantumNativeQNetworkConv):
    """Clone of `QuantumNativeQNetworkConv` that uses hardware-realizable variant of QuAK circuit."""
    
    @classmethod
    def monkeypatch_from_QuantumNativeQNetworkConv(cls, parent: QuantumNativeQNetworkConv) -> 'QuantumNativeQNetworkConvHardwareRealized':
        """Instantiate an instance of the `QuantumNativeQNetworkConvHardwareRealized` class using members from a base `QuantumNativeQNetworkConv` class.
        
        This is useful to reuse existing trained model parameters.
        """
        # Create new child instance for monkeypatching.
        monkey = cls(
            din=parent.din,
            n_actions=parent.n_actions,
            n_returns=parent.n_returns,
            z_min=parent.z_min,
            z_max=parent.z_max,
            m_moment=parent.m_moment,
            n_reuploads=parent.n_reuploads,
            n_layers=parent.n_layers,
            rngs_kwargs=parent.rngs_kwargs,
            pauli_enc=parent.pauli_enc,
            pauli_var=parent.pauli_var,
            imprimitive=parent.imprimitive,
            use_qft=parent.use_qft,
            input_preprocess=parent.input_preprocess,
            input_multiplex_activation=parent.input_multiplex_activation,
            w=parent.w,
            h=parent.h,
            frame_stack=parent.frame_stack,
            conv_douts=parent.conv_douts,
            conv_kernel_size=parent.conv_kernel_size,
            conv_strides=parent.conv_strides,
        )
        
        # Extract the graph state and variables from the parent and update the monkey child.
        graphdef, state = nnx.split(parent)
        nnx.update(monkey, state)
        
        # Return the updated monkey child.
        return monkey
    
    @nnx.jit(static_argnames=['n_reuploads'])
    def call_gen_action_dist(self, x, n_reuploads: int | None = None): # (batch,din,)
        """Generate action distribuion from input observation using hardware-realized QuAK circuit."""
        
        # Project input observation into shape that is compatible with the quantum circuit.
        x = self.project_observations(x) # (batch, n_layers, d_returns, 1,)
        
        # Track the batch size.
        batch_size = x.shape[0]
        
        # Prepare the n-th power of the m-th moment of the returns distribution.
        key = self.rngs.default() # Generate new random key for circuit measurement.
        # Initially set all actions to equal probability (here all have magnitude of 1, which will be normalized within the quantum circuit code later).
        probs_actions = jnp.ones(shape=(batch_size, self.n_actions,), dtype='float64')
        if n_reuploads is None:
            n_reuploads = self.n_reuploads
        # Create a batch of keys.
        subkeys = [key]
        for _ in range(n_reuploads):
            # Create a batch of keys for the current reupload.
            subkeys = jax.random.split(key=subkeys[0], num=batch_size)
            # Run the circuit on batched input.
            probs_actions = batched_run_circuit_action_dist_hardware_realizable( # (batch,2**d_actions,)
                # All device and QNode parameters.
                key=subkeys,
                device_name="default.qubit",
                shots=None,
                diff_method=None,
                # Circuit arguments.
                inputs=x,
                probs_actions=probs_actions,
                weights=self.circ_weights_var,
                n_actions=self.n_actions,
                d_actions=self.d_actions,
                d_returns=self.d_returns,
                m_moment=self.m_moment,
                n_layers=self.n_layers,
                U_atoms=self.U_atoms,
                pauli_enc=self.pauli_enc,
                pauli_var=self.pauli_var,
                imprimitive=self.imprimitive,
                use_qft=self.use_qft,
            )
            # Filter circuit output based on number of "active" actions.
            # If desired number of actions is not a multiple of 2, then filter out the excess actions on the qubit register (i.e., more actions can be represented on the register and we don't care about them).
            probs_actions = probs_actions[:,jnp.array(self.active_action_idx, dtype=int)] # (batch,n_actions,)
            # Ensure probabilities are normalized after the filter.
            probs_actions = probs_actions / probs_actions.sum(axis=-1, keepdims=True)
        return probs_actions

In [2]:
from pathlib import Path

In [3]:
root_sweeps = Path("../../wandb-sweep-data-qnrl/")
root_sweeps.mkdir(parents=True, exist_ok=True)

In [4]:
sweep_name_id = { # name: id
    # QnRL.
    'qnrl/FrozenLake-v1': '7yoeyqqr',
    'qnrl/CliffWalking-v1': 'nin93kq8',
    'qnrl/Acrobot-v1': '5ehazde3',
    'qnrl/CartPole-v1': 'qojis1fw',

    # C51.
    'c51/FrozenLake-v1': 'jzduwzvg',
    'c51/CliffWalking-v1': '8xyt1eb5',
    'c51/Acrobot-v1': 'n8imxz1v',
    'c51/CartPole-v1': 'arv708xs',
    
    # Breakout-v5.
    'qnrl/Breakout-v5': 'eigl7wk5',
    'c51/Breakout-v5': 'dg4sodfc',
    
    # SpaceInvaders-v5.
    'qnrl/SpaceInvaders-v5': '8adr9d1k',
    'c51/SpaceInvaders-v5': 'wbmr06xp',
}
best_runs_for_sweep = { # name: [id]
    # QnRL.
    "qnrl/FrozenLake-v1": ['xy1qeozs', 'nrg58af6', 'ohbnwer4', '1w6f17u8'],
    "qnrl/CliffWalking-v1": ['84ly3kb8', 'q14zmjpp', 'ujlm00gt', 'er4d6rnz'],
    "qnrl/Acrobot-v1": ['k9tdwgod', '7iae54v0', 'n7t9tfkv', 'se6p60c1'],
    "qnrl/CartPole-v1": ['0jash5pr', '8pi89rle', 'zds1hi8i', 'tqeh89te'],
    
    # C51.
    "c51/FrozenLake-v1": ['907k571g', 'i8w40ma5', '8p7uzbpv', '8oqjux3k'],
    "c51/CliffWalking-v1": ['zko23iy1', 'd962vucv', 'azhg19ww', '0nn68l3j'],
    "c51/Acrobot-v1": ['zbzemhwh', 'hs1wv6fh', 'ec4ti73g', 'y18igax1'],
    "c51/CartPole-v1": ['ubxwaw3p', 'f20h31mw', 'gdixmx80', '9eihhl7r'],
    
    # Breakout-v5.
    'qnrl/Breakout-v5': ['i8ijgd1m', 'syrhxro1', '4pc6bthv', '3pv6knud'],
    'c51/Breakout-v5': ['bxdtzg0b', 'cmn3go73', '62raq6de', '2vxp36ul'],
    
    # SpaceInvaders-v5.
    'qnrl/SpaceInvaders-v5': ['kiwbzgdv', '4kkrr3v5', '2artbn9x', 'nvmd7j1x'],
    'c51/SpaceInvaders-v5': ['wq1meahd', 'gbkustul', 'q75ssoqq', 'ekmn2epv'],
}

In [5]:
import glob
import json
import ast

import qnrl

def get_run_metadata(metadata_file: Path):
    # Load metadata as JSON dictionary.
    with open(str(metadata_file), 'r') as f:
        metadata = json.load(f)
    return metadata

def get_run_args(metadata_file: Path) -> dict:
    metadata = get_run_metadata(metadata_file)
    l: list[tuple[str, str]] = [tokens.split('=', 1) for tokens in metadata['args'] if '--' in tokens and '=' in tokens]
    d = dict((key.replace('--', '', 1), ast.literal_eval(val)) for key, val in l)
    return d

def loader(sweep_id: str, run_id: str) -> qnrl.algorithms.algorithm.Algorithm:

    dir_run = root_sweeps/f"sweep-{sweep_id}"/f"run-{run_id}"
    metadata_file = dir_run/"wandb-metadata.json"
    
    arg_dict = get_run_args(metadata_file)
    
    cfg = arg_dict['config']
    algo_config = qnrl.algorithms.algorithm.Algorithm.from_dict(cfg)
    
    # Load parameters from model save.
    model_dirs = {Path(root).stem: Path(root) for root in glob.glob(str(dir_run/"models"/"*"))}
    algo_config.load(model_dirs)
    
    return algo_config

In [6]:
### Now verify multiple environments.

def verify_sweep_run(
    sweep_id: str,
    run_id: str,
    samples: int = 1_000,
    seed: int = 42,
    ):
    """Create a hardware-realized variant of the quantum-native model in the given sweep-run and verify that it produces the same results as the original model over multiple random environment observation samples."""
    
    # Load algorithm instance from disk.
    alg: qnrl.algorithms.qnrl.QnRL = loader(
        sweep_id=sweep_id,
        run_id=run_id,
    )
    
    # Original trained model using original QuAK.
    model_quak = alg.model
    
    # Monkeypatched model using new hardware-realizable QuAK.
    match model_quak:
        case QuantumNativeQNetwork() if type(model_quak) is QuantumNativeQNetwork:
            model_monkey = QuantumNativeQNetworkHardwareRealized.monkeypatch_from_QuantumNativeQNetwork(model_quak)
        case QuantumNativeQNetworkConv() if type(model_quak) is QuantumNativeQNetworkConv:
            model_monkey = QuantumNativeQNetworkConvHardwareRealized.monkeypatch_from_QuantumNativeQNetworkConv(model_quak)
        case _:
            raise ValueError(f"Unsupported parent model class {type(model_quak)}")
    
    # Verify both the original model and the monkeypatched model have the same NNX parameters.
    _, state_quak = nnx.split(model_quak)
    _, state_monkey = nnx.split(model_monkey)
    is_same = jax.tree.map(
        lambda p1, p2: jnp.array_equal(p1, p2),
        state_quak, state_monkey,
    )
    all_equal = all(jnp.all(x) for x in jax.tree.leaves(is_same))
    assert all_equal, f"Original and monkeypatched parameter mismatch ({sweep_id=}, {run_id=})!"
    
    # Create environment.
    env = alg.env.make()
    
    # Set environment seed.
    env.observation_space.seed(seed)
    
    # Generate lots of random observation samples.
    obs_list = jnp.asarray([env.observation_space.sample() for _ in range(samples)])
    
    # Run all observation samples through the models.
    res_quak = jnp.asarray([model_quak(o) for o in obs_list]).squeeze()
    res_monkey = jnp.asarray([model_monkey(o) for o in obs_list]).squeeze()
    
    # Compare model output.
    flag_allmatch = jnp.allclose(res_quak, res_monkey)
    
    # Coalesce results and metadata into dictionaries.
    res = {
        'flag_allmatch': flag_allmatch,
    }
    meta = {
        'sweep_id': sweep_id,
        'run_id': run_id,
        'seed': seed,
        'samples': samples,
    }
    return res, meta, (obs_list, res_quak, res_monkey)

In [7]:
from tqdm import tqdm

def do_verify(
    samples: int,
    seed: int,
    best_run_idx: int,
    ):
    # Verify action-probability distributions are the same.
    verify_results = {}
    sweep_names_to_verify = ["qnrl/Acrobot-v1", "qnrl/FrozenLake-v1", "qnrl/CartPole-v1", "qnrl/CliffWalking-v1", "qnrl/Breakout-v5", "qnrl/SpaceInvaders-v5"]
    with tqdm(sweep_names_to_verify, desc="Sweeps") as pbar:
        for sweep_name in pbar:
            verify_results[sweep_name] = verify_sweep_run(
                sweep_id=sweep_name_id[sweep_name],
                run_id=best_runs_for_sweep[sweep_name][best_run_idx],
                samples=samples,
                seed=seed,
            )
            
            # Report current result in the progress bar.
            res, meta, _ = verify_results[sweep_name]
            pbar.set_postfix({'sweep_name': sweep_name, **res, **meta})
    return verify_results

def report_verify(verify_results: dict):
    # Report matching results.
    for sweep_name, tup in verify_results.items():
        res, meta, _ = tup
        print(f"{sweep_name}: {'✅' if res['flag_allmatch'] else '⛔️'} (seed={meta['seed']}, samples={meta['samples']}, sweep_id={meta['sweep_id']}, run_id={meta['run_id']})")

In [8]:
samples = 10_000
verify_results_runs = {}

In [9]:
key = 'idx=0,seed=42'
verify_results_runs[key] = do_verify(
    samples=samples,
    seed=42,
    best_run_idx=0,
)

print("Report:")
report_verify(verify_results_runs[key])

Sweeps:  67%|██████▋   | 4/6 [05:03<02:20, 70.29s/it, sweep_name=qnrl/CliffWalking-v1, flag_allmatch=True, sweep_id=nin93kq8, run_id=84ly3kb8, seed=42, samples=1e+4]A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
Sweeps: 100%|██████████| 6/6 [14:25<00:00, 144.27s/it, sweep_name=qnrl/SpaceInvaders-v5, flag_allmatch=True, sweep_id=8adr9d1k, run_id=kiwbzgdv, seed=42, samples=1e+4]

Report:
qnrl/Acrobot-v1: ✅ (seed=42, samples=10000, sweep_id=5ehazde3, run_id=k9tdwgod)
qnrl/FrozenLake-v1: ✅ (seed=42, samples=10000, sweep_id=7yoeyqqr, run_id=xy1qeozs)
qnrl/CartPole-v1: ✅ (seed=42, samples=10000, sweep_id=qojis1fw, run_id=0jash5pr)
qnrl/CliffWalking-v1: ✅ (seed=42, samples=10000, sweep_id=nin93kq8, run_id=84ly3kb8)
qnrl/Breakout-v5: ✅ (seed=42, samples=10000, sweep_id=eigl7wk5, run_id=i8ijgd1m)
qnrl/SpaceInvaders-v5: ✅ (seed=42, samples=10000, sweep_id=8adr9d1k, run_id=kiwbzgdv)


In [10]:
key = 'idx=-1,seed=42'
verify_results_runs[key] = do_verify(
    samples=samples,
    seed=42,
    best_run_idx=-1,
)

print("Report:")
report_verify(verify_results_runs[key])

Sweeps: 100%|██████████| 6/6 [15:20<00:00, 153.37s/it, sweep_name=qnrl/SpaceInvaders-v5, flag_allmatch=True, sweep_id=8adr9d1k, run_id=nvmd7j1x, seed=42, samples=1e+4]

Report:
qnrl/Acrobot-v1: ✅ (seed=42, samples=10000, sweep_id=5ehazde3, run_id=se6p60c1)
qnrl/FrozenLake-v1: ✅ (seed=42, samples=10000, sweep_id=7yoeyqqr, run_id=1w6f17u8)
qnrl/CartPole-v1: ✅ (seed=42, samples=10000, sweep_id=qojis1fw, run_id=tqeh89te)
qnrl/CliffWalking-v1: ✅ (seed=42, samples=10000, sweep_id=nin93kq8, run_id=er4d6rnz)
qnrl/Breakout-v5: ✅ (seed=42, samples=10000, sweep_id=eigl7wk5, run_id=3pv6knud)
qnrl/SpaceInvaders-v5: ✅ (seed=42, samples=10000, sweep_id=8adr9d1k, run_id=nvmd7j1x)


In [11]:
key = 'idx=0,seed=84'
verify_results_runs[key] = do_verify(
    samples=samples,
    seed=84,
    best_run_idx=0,
)

print("Report:")
report_verify(verify_results_runs[key])

Sweeps: 100%|██████████| 6/6 [10:03<00:00, 100.57s/it, sweep_name=qnrl/SpaceInvaders-v5, flag_allmatch=True, sweep_id=8adr9d1k, run_id=kiwbzgdv, seed=84, samples=1e+4]

Report:
qnrl/Acrobot-v1: ✅ (seed=84, samples=10000, sweep_id=5ehazde3, run_id=k9tdwgod)
qnrl/FrozenLake-v1: ✅ (seed=84, samples=10000, sweep_id=7yoeyqqr, run_id=xy1qeozs)
qnrl/CartPole-v1: ✅ (seed=84, samples=10000, sweep_id=qojis1fw, run_id=0jash5pr)
qnrl/CliffWalking-v1: ✅ (seed=84, samples=10000, sweep_id=nin93kq8, run_id=84ly3kb8)
qnrl/Breakout-v5: ✅ (seed=84, samples=10000, sweep_id=eigl7wk5, run_id=i8ijgd1m)
qnrl/SpaceInvaders-v5: ✅ (seed=84, samples=10000, sweep_id=8adr9d1k, run_id=kiwbzgdv)


In [12]:
key = 'idx=-1,seed=84'
verify_results_runs[key] = do_verify(
    samples=samples,
    seed=84,
    best_run_idx=-1,
)

print("Report:")
report_verify(verify_results_runs[key])

Sweeps: 100%|██████████| 6/6 [10:05<00:00, 101.00s/it, sweep_name=qnrl/SpaceInvaders-v5, flag_allmatch=True, sweep_id=8adr9d1k, run_id=nvmd7j1x, seed=84, samples=1e+4]

Report:
qnrl/Acrobot-v1: ✅ (seed=84, samples=10000, sweep_id=5ehazde3, run_id=se6p60c1)
qnrl/FrozenLake-v1: ✅ (seed=84, samples=10000, sweep_id=7yoeyqqr, run_id=1w6f17u8)
qnrl/CartPole-v1: ✅ (seed=84, samples=10000, sweep_id=qojis1fw, run_id=tqeh89te)
qnrl/CliffWalking-v1: ✅ (seed=84, samples=10000, sweep_id=nin93kq8, run_id=er4d6rnz)
qnrl/Breakout-v5: ✅ (seed=84, samples=10000, sweep_id=eigl7wk5, run_id=3pv6knud)
qnrl/SpaceInvaders-v5: ✅ (seed=84, samples=10000, sweep_id=8adr9d1k, run_id=nvmd7j1x)
